In [1]:
import numpy as np
import torch
import torchvision
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import sys

from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from torchsummary import summary
from torchvision import datasets, transforms

GRU and LSTM

In [11]:
input_size = 9
hidden_size = 16
num_layers = 2

# create an LSTM instance
lstm = nn.LSTM(input_size, hidden_size, num_layers)
print(lstm)

gru = nn.GRU(input_size, hidden_size, num_layers)
print(gru)

LSTM(9, 16, num_layers=2)
GRU(9, 16, num_layers=2)


In [12]:
# set data parameters
sequence_length = 5
batch_size = 2

# create random data
X = torch.rand(sequence_length, batch_size, input_size)

# create initial hidden and cell states
h0 = torch.zeros(num_layers, batch_size, hidden_size)
c0 = torch.zeros(num_layers, batch_size, hidden_size)

In [14]:
# forward pass through the LSTM
output, (hn, cn) = lstm(X, (h0, c0))

print("Input shape:", X.shape)
print("Hidden state shape:", hn.shape)
print("Cell state shape:", cn.shape)
print("Output shape:", output.shape)

Input shape: torch.Size([5, 2, 9])
Hidden state shape: torch.Size([2, 2, 16])
Cell state shape: torch.Size([2, 2, 16])
Output shape: torch.Size([5, 2, 16])


In [13]:
# forward pass through the GRU
output, hn = gru(X, h0)

print("Input shape:", X.shape)
print("Hidden state shape:", hn.shape)
print("Output shape:", output.shape)

Input shape: torch.Size([5, 2, 9])
Hidden state shape: torch.Size([2, 2, 16])
Output shape: torch.Size([5, 2, 16])


In [ ]:
for name, p in lstm.named_parameters():
    print(name, p.shape)
    # size is 64 because of the 4 gates concatenated (input, forget, cell, output) and the hidden size of 16

weight_ih_l0 torch.Size([64, 9])
weight_hh_l0 torch.Size([64, 16])
bias_ih_l0 torch.Size([64])
bias_hh_l0 torch.Size([64])
weight_ih_l1 torch.Size([64, 16])
weight_hh_l1 torch.Size([64, 16])
bias_ih_l1 torch.Size([64])
bias_hh_l1 torch.Size([64])


In [16]:
for name, p in gru.named_parameters():
    print(name, p.shape)
    # size is 48 because of the 3 gates concatenated (reset, update, new) and the hidden size of 16

weight_ih_l0 torch.Size([48, 9])
weight_hh_l0 torch.Size([48, 16])
bias_ih_l0 torch.Size([48])
bias_hh_l0 torch.Size([48])
weight_ih_l1 torch.Size([48, 16])
weight_hh_l1 torch.Size([48, 16])
bias_ih_l1 torch.Size([48])
bias_hh_l1 torch.Size([48])


In [ ]:
class LSTMModel(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, output_size=1):
        super(LSTMModel, self).__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers)
        self.fc = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        lstm_out, (hn, cn) = self.lstm(x)
        out = self.fc(lstm_out[-1])  # take the output of the last time step
        return out, hn, cn

In [10]:
# test the model with random data
model = LSTMModel(input_size, hidden_size, num_layers, output_size=1)
X = torch.rand(sequence_length, batch_size, input_size)
output, hn, cn = model(X)
print("Model input shape:", X.shape)
print("Model hidden state shape:", hn.shape)
print("Model cell state shape:", cn.shape)
print("Model output shape:", output.shape)

Model input shape: torch.Size([5, 2, 9])
Model hidden state shape: torch.Size([2, 2, 16])
Model cell state shape: torch.Size([2, 2, 16])
Model output shape: torch.Size([5, 2, 1])
